# Extract PBIR Report Metadata
Extract metadata from **PBIR reports** using sempy/semantic-link-labs only.

This notebook focuses on report objects that are readable via `connect_report`, and writes the extracted results into **Delta tables in the lakehouse**.

Primary output tables:
- `lineage_reports`
- `lineage_report_dataset_metadata`
- `lineage_report_pages`
- `lineage_report_visuals`
- `lineage_report_custom_visuals`
- `lineage_report_filters_report`
- `lineage_report_filters_page`
- `lineage_report_filters_visual`
- `lineage_report_visual_objects`
- `lineage_report_bookmarks`
- `lineage_report_level_measures`
- `lineage_report_semantic_model_objects`
- `lineage_report_visual_calculations`
- `lineage_report_extraction_log`

## 1. Imports

**Note**: Packages loaded from Fabric Environment (no installation needed).

In [ ]:
# Packages from attached Fabric Environment
import json
from datetime import datetime
from typing import Any, Dict, List
import sempy_labs.report
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from sempy.fabric import list_reports
from sempy_labs.report import connect_report

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.builder.getOrCreate()

print("✅ Imports successful (PBIR + sempy + spark)")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 19, Finished, Available, Finished, False)

✅ Imports successful (from environment)


## 2. Configuration

In [ ]:
# These will be provided as parameters when triggered via Livy
# For testing, set manually:
WORKSPACE_IDS = ["2b1d454b-61a1-4abb-96c3-2b476d945a96"]  # Can extract from multiple workspaces
LAKEHOUSE_ID = "8f644af3-54bc-4525-8cf8-7b78e5b01cd5"

EXTRACT_DETAILED_VISUALS = True
WRITE_MODE = "append"  # use "overwrite" for clean reruns during development

print(f"Extracting from {len(WORKSPACE_IDS)} workspace(s)")
print(f"Delta write mode: {WRITE_MODE}")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 20, Finished, Available, Finished, False)

Extracting from 1 workspace(s)


## 3. PBIR Extraction Functions (sempy only)

In [ ]:
def _to_records(value: Any) -> List[Dict[str, Any]]:
    """Normalize sempy outputs (mostly DataFrames) to list-of-dicts."""
    if value is None:
        return []

    if hasattr(value, "to_dict"):
        try:
            return value.to_dict(orient="records")
        except Exception:
            pass

    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]

    return [{"value": str(value)}]


def _is_pbir_format_required_error(error_message: str) -> bool:
    return "This ReportWrapper function requires the report to be in the PBIR format" in error_message


def _run_optional_call(
    rpt: Any,
    method_name: str,
    enabled: bool = True
 ) -> tuple[List[Dict[str, Any]], str | None, bool]:
    if not enabled:
        return [], None, False

    try:
        method = getattr(rpt, method_name, None)
        if method is None:
            return [], f"Method not available: {method_name}", False

        raw = method()
        records = _to_records(raw)
        return records, None, False
    except Exception as error:
        error_message = str(error)
        if _is_pbir_format_required_error(error_message):
            return [], error_message, True
        return [], error_message, False


def _serialize_value(value: Any) -> Any:
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, datetime):
        return value.isoformat()
    return json.dumps(value, default=str)


def _sanitize_column_name(name: str) -> str:
    """Convert arbitrary keys (e.g. 'Alt Text') to Delta-safe column names."""
    sanitized_chars = []
    for ch in name.strip():
        if ch.isalnum() or ch == "_":
            sanitized_chars.append(ch.lower())
        else:
            sanitized_chars.append("_")

    sanitized = "".join(sanitized_chars)
    while "__" in sanitized:
        sanitized = sanitized.replace("__", "_")
    sanitized = sanitized.strip("_")

    if not sanitized:
        sanitized = "col"
    if sanitized[0].isdigit():
        sanitized = f"col_{sanitized}"

    return sanitized


def _add_sanitized_row_fields(prepared_row: Dict[str, Any], raw_row: Dict[str, Any]) -> None:
    """Add row fields with stable, unique, Delta-safe names."""
    used_names = set(prepared_row.keys())

    for raw_key, raw_value in raw_row.items():
        base_name = _sanitize_column_name(str(raw_key))
        final_name = base_name
        suffix = 2

        while final_name in used_names:
            final_name = f"{base_name}_{suffix}"
            suffix += 1

        prepared_row[final_name] = _serialize_value(raw_value)
        used_names.add(final_name)


def _prepare_rows(
    rows: List[Dict[str, Any]],
    workspace_id: str,
    report_id: str,
    report_name: str,
    extraction_timestamp: str
 ) -> List[Dict[str, Any]]:
    prepared_rows = []

    for row in rows:
        prepared_row = {
            "workspace_id": workspace_id,
            "report_id": report_id,
            "report_name": report_name,
            "extraction_timestamp": extraction_timestamp
        }

        _add_sanitized_row_fields(prepared_row, row)
        prepared_rows.append(prepared_row)

    return prepared_rows


def _table_path(table_name: str) -> str:
    # Kept for logging only; writes now use managed Delta tables in catalog.
    return f"Tables/{table_name}"


def _delta_table_exists(table_name: str) -> bool:
    return bool(spark.catalog.tableExists(table_name))


def _load_delta_table(table_name: str):
    return spark.table(table_name)


def _drop_table_if_exists(table_name: str) -> None:
    if _delta_table_exists(table_name):
        spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")


def _register_delta_table(table_name: str) -> None:
    # No-op because saveAsTable registers managed table metadata automatically.
    return None


def _prepare_table_location_for_write(table_name: str, mode: str) -> None:
    # For managed tables, explicit cleanup is only needed for overwrite semantics.
    if mode == "overwrite":
        _drop_table_if_exists(table_name)


def _build_report_row(
    report_id: str,
    workspace_id: str,
    report_name: str,
    dataset_id: Any,
    report_info_row: Dict[str, Any],
    extraction_timestamp: str,
    duration: float,
    method_errors: Dict[str, str],
    extract_detailed: bool,
    counts: Dict[str, int]
 ) -> Dict[str, Any]:
    report_row = {
        "workspace_id": workspace_id,
        "report_id": report_id,
        "report_name": report_name,
        "display_name": report_info_row.get("Display Name") or report_info_row.get("DisplayName") or report_name,
        "dataset_id": dataset_id,
        "extraction_timestamp": extraction_timestamp,
        "extraction_method": "sempy_labs.report (PBIR)",
        "extraction_duration": duration,
        "pbir_only": True,
        "detailed_extraction": extract_detailed,
        "skipped_methods": json.dumps(method_errors, default=str),
        "raw_report_info": json.dumps(report_info_row, default=str)
    }

    for key, value in counts.items():
        report_row[f"count_{key}"] = value

    return report_row


def _build_dataset_metadata_row(
    report_id: str,
    workspace_id: str,
    report_name: str,
    dataset_id: Any,
    report_info_row: Dict[str, Any],
    extraction_timestamp: str
 ) -> Dict[str, Any]:
    """Flatten base report metadata from sempy_labs.report.get_report()."""
    row = {
        "workspace_id": workspace_id,
        "report_id": report_id,
        "report_name": report_name,
        "dataset_id": dataset_id,
        "extraction_timestamp": extraction_timestamp
    }
    _add_sanitized_row_fields(row, report_info_row)
    return row


def _build_log_rows(
    workspace_id: str,
    report_id: str,
    report_name: str,
    extraction_timestamp: str,
    status: str,
    skip_reason: str | None = None,
    error_message: str | None = None,
    method_errors: Dict[str, str] | None = None
 ) -> List[Dict[str, Any]]:
    rows = []

    if method_errors:
        for method_name, method_error in method_errors.items():
            rows.append({
                "workspace_id": workspace_id,
                "report_id": report_id,
                "report_name": report_name,
                "extraction_timestamp": extraction_timestamp,
                "status": status,
                "scope": "method",
                "name": method_name,
                "skip_reason": "method_failed",
                "message": method_error
            })

    if error_message:
        rows.append({
            "workspace_id": workspace_id,
            "report_id": report_id,
            "report_name": report_name,
            "extraction_timestamp": extraction_timestamp,
            "status": status,
            "scope": "report",
            "name": report_id,
            "skip_reason": skip_reason,
            "message": error_message
        })

    return rows


def _append_to_delta(table_name: str, rows: List[Dict[str, Any]], mode: str = "append") -> int:
    if not rows:
        return 0

    _prepare_table_location_for_write(table_name, mode)

    df = spark.createDataFrame(rows)

    writer = (
        df.write
        .format("delta")
        .option("mergeSchema", "true")
    )

    if mode == "overwrite":
        writer.mode("overwrite").saveAsTable(table_name)
    else:
        writer.mode("append").saveAsTable(table_name)

    return len(rows)


def extract_report_metadata(report_id: str, workspace_id: str, extract_detailed: bool = True) -> dict:
    """
    PBIR-focused extraction using sempy_labs.report APIs only.

    Rules:
    - If error is PBIR-format-required, skip the whole report.
    - If only one list_* function fails for another reason, skip that function and continue.
    """
    print(f"\n📊 Extracting report: {report_id}")
    extraction_start = datetime.now()

    try:
        report_df = sempy_labs.report.get_report(workspace=workspace_id, report=report_id)
        report_rows = _to_records(report_df)
        report_info_row = report_rows[0] if report_rows else {}

        with connect_report(report=report_id, workspace=workspace_id) as rpt:
            method_errors: Dict[str, str] = {}

            method_map = [
                ("pages", "list_pages"),
                ("visuals", "list_visuals"),
                ("customVisuals", "list_custom_visuals"),
                ("reportFilters", "list_report_filters"),
                ("pageFilters", "list_page_filters"),
                ("visualFilters", "list_visual_filters"),
                ("visualObjects", "list_visual_objects"),
                ("bookmarks", "list_bookmarks"),
                ("reportLevelMeasures", "list_report_level_measures"),
                ("semanticModelObjects", "list_semantic_model_objects"),
                ("visualCalculations", "list_visual_calculations")
            ]

            records_by_key: Dict[str, List[Dict[str, Any]]] = {}

            for key, method_name in method_map:
                records, error, requires_pbir = _run_optional_call(
                    rpt=rpt,
                    method_name=method_name,
                    enabled=extract_detailed
                )

                if requires_pbir:
                    raise ValueError(f"PBIR_REQUIRED::{method_name}::{error}")

                records_by_key[key] = records
                if error:
                    method_errors[method_name] = error

            extraction_end = datetime.now()
            extraction_timestamp = extraction_end.isoformat()
            duration = (extraction_end - extraction_start).total_seconds()

            report_name = (
                report_info_row.get("Name")
                or report_info_row.get("Display Name")
                or report_info_row.get("DisplayName")
                or str(report_id)
            )
            dataset_id = report_info_row.get("Dataset Id") or report_info_row.get("DatasetId")

            if method_errors:
                print("  ⚠️  Skipped failing sempy methods:")
                for method_name, method_error in method_errors.items():
                    print(f"     - {method_name}: {method_error}")

            counts = {
                "datasetMetadata": 1,
                "pages": len(records_by_key.get("pages", [])),
                "visuals": len(records_by_key.get("visuals", [])),
                "customVisuals": len(records_by_key.get("customVisuals", [])),
                "reportFilters": len(records_by_key.get("reportFilters", [])),
                "pageFilters": len(records_by_key.get("pageFilters", [])),
                "visualFilters": len(records_by_key.get("visualFilters", [])),
                "visualObjects": len(records_by_key.get("visualObjects", [])),
                "bookmarks": len(records_by_key.get("bookmarks", [])),
                "reportLevelMeasures": len(records_by_key.get("reportLevelMeasures", [])),
                "semanticModelObjects": len(records_by_key.get("semanticModelObjects", [])),
                "visualCalculations": len(records_by_key.get("visualCalculations", []))
            }

            print(f"  ✅ Success - {report_name}")
            print(f"  ℹ️  Pages: {counts['pages']}, Visuals: {counts['visuals']}")

            table_rows = {
                "lineage_reports": [_build_report_row(
                    report_id=report_id,
                    workspace_id=workspace_id,
                    report_name=report_name,
                    dataset_id=dataset_id,
                    report_info_row=report_info_row,
                    extraction_timestamp=extraction_timestamp,
                    duration=duration,
                    method_errors=method_errors,
                    extract_detailed=extract_detailed,
                    counts=counts
                )],
                "lineage_report_dataset_metadata": [_build_dataset_metadata_row(
                    report_id=report_id,
                    workspace_id=workspace_id,
                    report_name=report_name,
                    dataset_id=dataset_id,
                    report_info_row=report_info_row,
                    extraction_timestamp=extraction_timestamp
                )],
                "lineage_report_pages": _prepare_rows(records_by_key.get("pages", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_visuals": _prepare_rows(records_by_key.get("visuals", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_custom_visuals": _prepare_rows(records_by_key.get("customVisuals", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_filters_report": _prepare_rows(records_by_key.get("reportFilters", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_filters_page": _prepare_rows(records_by_key.get("pageFilters", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_filters_visual": _prepare_rows(records_by_key.get("visualFilters", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_visual_objects": _prepare_rows(records_by_key.get("visualObjects", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_bookmarks": _prepare_rows(records_by_key.get("bookmarks", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_level_measures": _prepare_rows(records_by_key.get("reportLevelMeasures", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_semantic_model_objects": _prepare_rows(records_by_key.get("semanticModelObjects", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_visual_calculations": _prepare_rows(records_by_key.get("visualCalculations", []), workspace_id, report_id, report_name, extraction_timestamp),
                "lineage_report_extraction_log": _build_log_rows(
                    workspace_id=workspace_id,
                    report_id=report_id,
                    report_name=report_name,
                    extraction_timestamp=extraction_timestamp,
                    status="success",
                    method_errors=method_errors
                )
            }

            return {
                "artifactId": report_id,
                "artifactType": "Report",
                "workspaceId": workspace_id,
                "timestamp": extraction_timestamp,
                "reportName": report_name,
                "status": "success",
                "counts": counts,
                "tableRows": table_rows
            }

    except Exception as error:
        extraction_end = datetime.now()
        extraction_timestamp = extraction_end.isoformat()
        duration = (extraction_end - extraction_start).total_seconds()
        error_message = str(error)

        report_name = str(report_id)
        if _is_pbir_format_required_error(error_message) or error_message.startswith("PBIR_REQUIRED::"):
            print("  ⚠️  Skipping whole report (requires PBIR format)")
            skip_reason = "requires_pbir_format"
        else:
            print(f"  ⚠️  Skipping report (unreadable via sempy): {error_message}")
            skip_reason = "unreadable_via_sempy"

        return {
            "artifactId": report_id,
            "artifactType": "Report",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "reportName": report_name,
            "status": "skipped",
            "counts": {},
            "tableRows": {
                "lineage_report_extraction_log": _build_log_rows(
                    workspace_id=workspace_id,
                    report_id=report_id,
                    report_name=report_name,
                    extraction_timestamp=extraction_timestamp,
                    status="skipped",
                    skip_reason=skip_reason,
                    error_message=error_message
                )
            },
            "metadata": {
                "skipReason": skip_reason,
                "extractionMethod": "sempy_labs.report (PBIR)",
                "pbirOnly": True,
                "extractionDuration": duration,
                "errorMessage": error_message
            }
        }


print("✅ PBIR extraction functions ready")
print("   • sempy_labs.report.get_report")
print("   • connect_report(...).list_pages")
print("   • connect_report(...).list_visuals")
print("   • connect_report(...).list_custom_visuals")
print("   • connect_report(...).list_report_filters")
print("   • connect_report(...).list_page_filters")
print("   • connect_report(...).list_visual_filters")
print("   • connect_report(...).list_visual_objects")
print("   • connect_report(...).list_bookmarks")
print("   • connect_report(...).list_report_level_measures")
print("   • connect_report(...).list_semantic_model_objects")
print("   • connect_report(...).list_visual_calculations")
print("   • output is written as managed Delta tables")
print("   • source field names are sanitized to Delta-safe column names")
print("   • base metadata is written to lineage_report_dataset_metadata")

: 

## 4. Process All Workspaces

In [ ]:
# Initialize results tracking
all_results = []
table_write_counts: Dict[str, int] = {}

EXPECTED_TABLES = [
    "lineage_reports",
    "lineage_report_dataset_metadata",
    "lineage_report_pages",
    "lineage_report_visuals",
    "lineage_report_custom_visuals",
    "lineage_report_filters_report",
    "lineage_report_filters_page",
    "lineage_report_filters_visual",
    "lineage_report_visual_objects",
    "lineage_report_bookmarks",
    "lineage_report_level_measures",
    "lineage_report_semantic_model_objects",
    "lineage_report_visual_calculations",
    "lineage_report_extraction_log"
]

# Optional reset for clean reruns while staying in managed-table mode.
if WRITE_MODE == "overwrite":
    print("🧹 WRITE_MODE=overwrite: dropping existing managed tables before extraction")
    for table_name in EXPECTED_TABLES:
        _drop_table_if_exists(table_name)

# Process each workspace
for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing workspace: {workspace_id}")

    try:
        reports_df = list_reports(workspace=workspace_id)
        print(f"  ✅ Found {len(reports_df)} report(s)")

        for _, report_row in reports_df.iterrows():
            report_id = report_row.get("Id") or report_row.get("id")
            report_name = report_row.get("Name") or report_row.get("DisplayName", "Unknown")

            if not report_id:
                print(f"  ⚠️  Skipping report '{report_name}' - no valid ID")
                continue

            result = extract_report_metadata(
                report_id=report_id,
                workspace_id=workspace_id,
                extract_detailed=EXTRACT_DETAILED_VISUALS
            )
            all_results.append(result)

            write_mode = WRITE_MODE
            for table_name, rows in result.get("tableRows", {}).items():
                written_count = _append_to_delta(table_name, rows, mode=write_mode)
                if written_count:
                    table_write_counts[table_name] = table_write_counts.get(table_name, 0) + written_count
                    print(f"    └─ Wrote {written_count} row(s) to {table_name}")

            if write_mode == "overwrite":
                write_mode = "append"

    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")

print(f"\n✅ Extraction complete: {len(all_results)} report(s) processed")
print("\nDelta rows written:")
for table_name in sorted(table_write_counts.keys()):
    print(f"   {table_name}: {table_write_counts[table_name]}")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 22, Finished, Available, Finished, False)


🔍 Processing workspace: 2b1d454b-61a1-4abb-96c3-2b476d945a96
  ✅ Found 2 report(s)
  📋 Columns: ['Id', 'Report Type', 'Format', 'Name', 'Web Url', 'Embed Url', 'Is From Pbix', 'Is Owned By Me', 'Dataset Id', 'App Id', 'Description', 'Dataset Workspace Id', 'Users', 'Subscriptions', 'Original Report Id']

📊 Extracting Report: a8247d4d-8590-4bd6-8521-0f84bbb73a77
  ℹ️  Report storage format: PBIR-Legacy
  🔄 Using Fabric REST API for non-PBIR report...
  ✅ REST API success - Report: Training Datamodel Solution
  ℹ️  Dataset: None
  🔍 Parsing report definition for pages and visuals...
    ⚠️  Could not decode part 'StaticResources/RegisteredResources/AdventureWorksLogo9335593362734207.jpg': 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
    └─ Page '1. Einfache Aggregation und Filter Context': 7 visual(s)
    └─ Saved: Files/lineage/raw/reports/a8247d4d-8590-4bd6-8521-0f84bbb73a77.json

📊 Extracting Report: 88e4eb9f-f94f-4618-9225-20921b8f614c
  ℹ️  Report storage 

## 5. Summary

In [ ]:
success_count = sum(1 for r in all_results if r.get("status") == "success")
skipped_count = sum(1 for r in all_results if r.get("status") == "skipped")
error_count = len(all_results) - success_count - skipped_count

print("\n" + "=" * 50)
print("EXTRACTION SUMMARY (PBIR + SEMPY + DELTA)")
print("=" * 50)
print(f"Total Reports: {len(all_results)}")
print(f"✅ Success: {success_count}")
print(f"⚠️  Skipped (non-PBIR/unreadable): {skipped_count}")
print(f"❌ Errors: {error_count}")
print("\nDelta tables written:")
for table_name in sorted(table_write_counts.keys()):
    print(f"- {table_name}: {table_write_counts[table_name]} row(s)")
print("=" * 50)

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 23, Finished, Available, Finished, False)


EXTRACTION SUMMARY
Total Reports: 2
✅ Full Success: 2 (with pages/visuals)
⚠️  Limited Success: 0 (basic metadata only)
❌ Errors: 0

Results saved to: Files/lineage/raw/reports/


---
# 🧪 Testing & Validation

## Test Single Report Before Full Extraction
Run the test cell below to verify extraction works correctly with one report before processing all reports in a workspace.

## 🧪 Test Single Report Extraction

In [ ]:
# TEST: Extract a single report and write its results to Delta tables

TEST_WORKSPACE_ID = "2b1d454b-61a1-4abb-96c3-2b476d945a96"
TEST_REPORT_ID = "a8247d4d-8590-4bd6-8521-0f84bbb73a77"

if not TEST_WORKSPACE_ID or not TEST_REPORT_ID:
    print("⚠️  Set TEST_WORKSPACE_ID and TEST_REPORT_ID before running this test.")
else:
    print("🔎 Testing PBIR extraction...\n")
    print(f"Workspace ID: {TEST_WORKSPACE_ID}")
    print(f"Report ID: {TEST_REPORT_ID}\n")

    test_result = extract_report_metadata(
        report_id=TEST_REPORT_ID,
        workspace_id=TEST_WORKSPACE_ID,
        extract_detailed=EXTRACT_DETAILED_VISUALS
    )

    print("\n" + "=" * 60)
    print("📋 EXTRACTION RESULT")
    print("=" * 60)
    print(f"Status: {test_result.get('status', 'N/A')}")
    print(f"Report: {test_result.get('reportName', 'N/A')}")
    for count_name, count_value in test_result.get("counts", {}).items():
        print(f"{count_name}: {count_value}")

    test_write_counts = {}
    test_mode = "overwrite" if WRITE_MODE == "overwrite" else "append"
    for table_name, rows in test_result.get("tableRows", {}).items():
        written_count = _append_to_delta(table_name, rows, mode=test_mode)
        if written_count:
            test_write_counts[table_name] = written_count
            print(f"Wrote {written_count} row(s) to {table_name}")
        if test_mode == "overwrite":
            test_mode = "append"

    print("=" * 60)

StatementMeta(, c49f1664-df42-4872-8c4f-85d6696b2159, 8, Finished, Available, Finished, False)

⚠️  Set TEST_WORKSPACE_ID and TEST_REPORT_ID before running this verification cell.


StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 24, Finished, Available, Finished, False)

🔎 Testing report extraction...

Workspace ID: 2b1d454b-61a1-4abb-96c3-2b476d945a96
Report ID: a8247d4d-8590-4bd6-8521-0f84bbb73a77

Storage Format: PBIR-Legacy
Expected Method: REST API with parsing


📊 Extracting Report: a8247d4d-8590-4bd6-8521-0f84bbb73a77
  ℹ️  Report storage format: PBIR-Legacy
  🔄 Using Fabric REST API for non-PBIR report...
  ✅ REST API success - Report: Training Datamodel Solution
  ℹ️  Dataset: None
  🔍 Parsing report definition for pages and visuals...
    ⚠️  Could not decode part 'StaticResources/RegisteredResources/AdventureWorksLogo9335593362734207.jpg': 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
    └─ Page '1. Einfache Aggregation und Filter Context': 7 visual(s)

📋 EXTRACTION RESULT
Status: success
Method: fabric_rest_api_with_definition_parsing
Format: PBIR-Legacy
Duration: 20.83s

Report: Training Datamodel Solution
Dataset: None
Pages: 1
Visuals: 7

📊 Visual Distribution:
   • 1. Einfache Aggregation und Filter Context: 7 

## 🧪 Verify Saved Data

In [ ]:
# Inspect the Delta tables written by this notebook
table_names = [
    "lineage_reports",
    "lineage_report_dataset_metadata",
    "lineage_report_pages",
    "lineage_report_visuals",
    "lineage_report_custom_visuals",
    "lineage_report_filters_report",
    "lineage_report_filters_page",
    "lineage_report_filters_visual",
    "lineage_report_visual_objects",
    "lineage_report_bookmarks",
    "lineage_report_level_measures",
    "lineage_report_semantic_model_objects",
    "lineage_report_visual_calculations",
    "lineage_report_extraction_log"
],

for table_name in table_names:
    try:
        if not _delta_table_exists(table_name):
            print(f"⚠️  {table_name}: table does not exist yet")
            continue

        row_count = _load_delta_table(table_name).count()
        print(f"📊 {table_name}: {row_count} row(s)")
        if row_count > 0:
            display(_load_delta_table(table_name).limit(5))
    except Exception as e:
        print(f"⚠️  {table_name}: not available yet ({e})")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 25, Finished, Available, Finished, False)

❌ No files found or error reading: path: /lakehouse/default/Files/lineage/raw/reports is invalid, if you want to operate on the local path, please add the prefix file:, for example file:/lakehouse/default/Files/lineage/raw/reports. Exception: Operation failed: "Bad Request", 400, GET, http://onelake.dfs.fabric.microsoft.com/a559a09d-159b-43f9-a5f7-f908bc5a77bb?upn=false&resource=filesystem&maxResults=5000&directory=lakehouse/default/Files/lineage/raw/reports&timeout=90&recursive=false, FriendlyNameSupportDisabled, "Request Failed with WorkspaceId and ArtifactId should be either valid Guids or valid Names"
   Run extraction first (cell 7)


## 🧪 Cross-Reference with Semantic Models

In [ ]:
# Analyze which reports use which semantic models from Delta tables
from pyspark.sql import functions as F

try:
    if not _delta_table_exists("lineage_reports"):
        print("❌ lineage_reports does not exist yet under Tables/lineage_reports")
    else:
        reports_df = _load_delta_table("lineage_reports")

        dataset_usage = (
            reports_df
            .filter(F.col("dataset_id").isNotNull())
            .select("dataset_id", "report_name")
            .distinct()
            .orderBy("dataset_id", "report_name")
        )

        rows = dataset_usage.collect()
        if rows:
            current_dataset = None
            current_reports = []

            print("📊 Dataset → Report Linkage:\n")
            for row in rows:
                if row["dataset_id"] != current_dataset:
                    if current_dataset is not None:
                        print(f"Dataset {current_dataset[:8]}... ({len(current_reports)} report(s)):")
                        for report_name in current_reports:
                            print(f"   └─ {report_name}")
                        print()
                    current_dataset = row["dataset_id"]
                    current_reports = [row["report_name"]]
                else:
                    current_reports.append(row["report_name"])

            if current_dataset is not None:
                print(f"Dataset {current_dataset[:8]}... ({len(current_reports)} report(s)):")
                for report_name in current_reports:
                    print(f"   └─ {report_name}")
                print()
        else:
            print("No dataset linkages found in lineage_reports")

except Exception as e:
    print(f"❌ Error analyzing linkages: {e}")

StatementMeta(, 9fc5af37-83d2-4e94-87a6-86a34db6b713, 26, Finished, Available, Finished, False)

❌ Error analyzing linkages: path: /lakehouse/default/Files/lineage/raw/reports is invalid, if you want to operate on the local path, please add the prefix file:, for example file:/lakehouse/default/Files/lineage/raw/reports. Exception: Operation failed: "Bad Request", 400, GET, http://onelake.dfs.fabric.microsoft.com/a559a09d-159b-43f9-a5f7-f908bc5a77bb?upn=false&resource=filesystem&maxResults=5000&directory=lakehouse/default/Files/lineage/raw/reports&timeout=90&recursive=false, FriendlyNameSupportDisabled, "Request Failed with WorkspaceId and ArtifactId should be either valid Guids or valid Names"


---
## Troubleshooting (PBIR + sempy + Delta)

### Common Issues

**1. Missing sempy packages**
- Attach the published Fabric Environment that includes `semantic-link` and `semantic-link-labs`.

**2. Report is skipped**
- This notebook is PBIR-focused and uses `connect_report` APIs.
- Non-PBIR reports or reports not readable via sempy are marked as `skipped` in `lineage_report_extraction_log`.

**3. Permission errors (401/403)**
- Ensure you have report/workspace access (Contributor recommended).

**4. Empty detail tables**
- Some reports do not contain all object types, so some Delta tables can be empty for a run.
- Per-method failures are logged and the remaining sempy methods still write their results.

### Storage behavior

- Uses only sempy APIs:
  - `sempy_labs.report.get_report(...)`
  - `connect_report(...).list_*` methods
- Writes structured results into Delta tables in the attached lakehouse.
- Uses `WRITE_MODE = "append"` by default; switch to `overwrite` for clean reruns during development.